# 4. VAR-CLIP Style Personalization: Paper-Faithful PFB + SAC

This notebook is a **text-prompt plus one style-reference** experiment. It deliberately does not accept a content image.

```text
text prompt                -> two identical VAR-CLIP inference streams
style reference image      -> frozen VAR VAE multi-scale feature pyramid
content stream             -> unmodified structural reference
stylized generation stream -> PFB at one pivotal step + SAC at later steps
```

This is the proper VAR-CLIP adaptation of *A Training-Free Style-Personalization via SVD-Based Feature Decomposition*.

It is **style-personalized text-to-image generation**, not content-image neural style transfer:

```text
input:  "a photo of a golden retriever sitting on grass" + dog sketch reference
output: a newly generated golden retriever with sketch-like appearance
```

Start with an ImageNet-friendly object such as a dog or mountain. Human portraits and crowded scenes are intentionally excluded from the default cases because VAR-CLIP-d16 is less reliable there.


## Method Contract: What Matches the Paper

Both paths receive the same prompt and begin from the same random seed. Before PFB they therefore follow the same ordinary VAR-CLIP trajectory.

```text
content stream:    unmodified generation with prompt T
stylized stream:   same prompt T, then PFB at pivotal feature F_s
style stream:      VAE encode the single style reference image
```

At the pivotal step:

```text
F_gen <- Phi(F_style) + (F_gen - Phi(F_gen))
```

At later transformer steps, SAC replaces only stylized-stream Q/K with Q/K from the unmodified content stream. Generated V remains unchanged.

```text
Q_gen <- Q_content
K_gen <- K_content
V_gen remains generated
```

The source paper identifies its third feature in Infinity-2B as pivotal. VAR-CLIP-d16 has a different backbone and ten scales, so this notebook treats step 3 as the starting hypothesis and includes a model-specific pivotal-step ablation.


In [ ]:
# Colab setup: Runtime -> Change runtime type -> T4 GPU (or better), then run this cell.
!nvidia-smi

import os
import subprocess
from pathlib import Path

assert os.path.exists('/usr/local/cuda') or os.environ.get('COLAB_GPU'), (
    'No GPU runtime detected. In Colab, select Runtime -> Change runtime type -> T4 GPU, then reconnect.'
)

if Path('/kaggle/working').exists():
    RUNTIME_ROOT = Path('/kaggle/working')
elif Path('/content').exists():
    RUNTIME_ROOT = Path('/content')
else:
    RUNTIME_ROOT = Path.cwd()

VAR_CLIP_REPO = 'https://github.com/daixiangzi/VAR-CLIP.git'
VAR_CLIP_DIR = RUNTIME_ROOT / 'VAR-CLIP'
OUTPUT_DIR = RUNTIME_ROOT / 'VAR_CLIP_outputs' / 'paper_faithful_pfb_sac'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not VAR_CLIP_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', VAR_CLIP_REPO, str(VAR_CLIP_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'fetch', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'reset', '--hard', 'origin/master'], check=True)

os.chdir(VAR_CLIP_DIR)
print('VAR-CLIP source:', VAR_CLIP_DIR)
print('Output directory:', OUTPUT_DIR)


In [ ]:
# Keep Colab's CUDA-enabled PyTorch. The package below provides the `open_clip` import
# required by the official VAR-CLIP repository.
!pip -q install gdown huggingface_hub einops typed-argument-parser pytz open_clip_torch pandas tqdm

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
assert torch.cuda.is_available(), 'A CUDA GPU is required.'


## Download Checkpoints

VAR-CLIP expects three weights:

```text
VAR VAE:        FoundationVision/var on Hugging Face
VAR-CLIP-d16:   official author checkpoint on Google Drive
CLIP ViT-L/14:  OpenAI checkpoint at pretrained/ViT-L-14.pt
```

The last local path is required by the upstream VAR-CLIP source.


In [ ]:
from huggingface_hub import hf_hub_download
import gdown

PRETRAINED_DIR = VAR_CLIP_DIR / 'pretrained'
LOCAL_OUTPUT_DIR = VAR_CLIP_DIR / 'local_output'
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vae_path = Path(hf_hub_download(
    repo_id='FoundationVision/var',
    filename='vae_ch160v4096z32.pth',
    local_dir=PRETRAINED_DIR,
))

clip_path = PRETRAINED_DIR / 'ViT-L-14.pt'
OPENAI_CLIP_VIT_L14_URL = (
    'https://openaipublic.azureedge.net/clip/models/'
    'b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt'
)
if not clip_path.exists() or clip_path.stat().st_size < 500_000_000:
    subprocess.run(['wget', '-c', '--show-progress', '-O', str(clip_path), OPENAI_CLIP_VIT_L14_URL], check=True)

var_clip_path = LOCAL_OUTPUT_DIR / 'ar-ckpt-last.pth'
VAR_CLIP_CHECKPOINT_URL = 'https://drive.google.com/file/d/10gSxvaKaNKJcnqFhU7hQywU28w3nbgoV/view?usp=sharing'
if not var_clip_path.exists() or var_clip_path.stat().st_size < 100_000_000:
    gdown.download(url=VAR_CLIP_CHECKPOINT_URL, output=str(var_clip_path), fuzzy=True)

assert vae_path.exists(), vae_path
assert clip_path.exists() and clip_path.stat().st_size > 500_000_000, 'Incomplete OpenAI CLIP download.'
assert var_clip_path.exists() and var_clip_path.stat().st_size > 100_000_000, 'Incomplete VAR-CLIP download.'
print('VAE:', vae_path)
print('CLIP:', clip_path)
print('VAR-CLIP:', var_clip_path)


## Load Frozen VAR-CLIP

The small source compatibility patch only addresses modern PyTorch’s default `weights_only=True` behavior for the trusted TorchScript CLIP checkpoint. It does not change VAR-CLIP weights or inference behavior.


In [ ]:
import gc
import math
import random
import sys
import importlib
import types
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm

# PyTorch 2.6+ compatibility for the trusted OpenAI TorchScript checkpoint expected by VAR-CLIP.
clip_source = VAR_CLIP_DIR / 'models' / 'clip.py'
clip_source_text = clip_source.read_text()
old_clip_call = "pretrained='pretrained/ViT-L-14.pt')"
new_clip_call = "pretrained='pretrained/ViT-L-14.pt', weights_only=False)"
if old_clip_call in clip_source_text:
    clip_source.write_text(clip_source_text.replace(old_clip_call, new_clip_call, 1))
assert 'weights_only=False' in clip_source.read_text()
if 'models.clip' in sys.modules:
    importlib.reload(sys.modules['models.clip'])

# Avoid unnecessary default initialization before loading the large checkpoints.
setattr(torch.nn.Linear, 'reset_parameters', lambda self: None)
setattr(torch.nn.LayerNorm, 'reset_parameters', lambda self: None)

from clip_util import CLIPWrapper
from models.clip import clip_vit_l14
from tokenizer import tokenize
from models import build_vae_var
from models.basic_var import slow_attn
from models.helpers import sample_with_top_k_top_p_

MODEL_DEPTH = 16
PATCH_NUMS = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)
device = 'cuda'

vae, var_clip = build_vae_var(
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    device=device,
    patch_nums=PATCH_NUMS,
    n_cond_embed=768,
    depth=MODEL_DEPTH,
    shared_aln=False,
)

clip_model = CLIPWrapper(clip_vit_l14(pretrained=True).to(device).eval(), normalize=True)
vae.load_state_dict(torch.load(vae_path, map_location='cpu'), strict=True)
checkpoint = torch.load(var_clip_path, map_location='cpu')
var_clip.load_state_dict(checkpoint['trainer']['var_wo_ddp'], strict=True)
vae.eval()
var_clip.eval()
for model in (vae, var_clip):
    for parameter in model.parameters():
        parameter.requires_grad_(False)

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
print('Frozen VAR-CLIP-d16 and CLIP ViT-L/14 are ready.')


## Prompt and Style Cases

Each case uses one easy content prompt and a style reference with compatible subject matter. The default is a dog prompt paired with a dog sketch; this minimizes source-subject leakage during the first correctness check.

The reference image is used only for PFB. It is never used as a content input or SAC teacher.


In [ ]:
STYLE_WORKSPACE_REPO = 'https://github.com/LeeHoang2710/Style-Transfer-Experiment.git'
STYLE_WORKSPACE = RUNTIME_ROOT / 'VAR_Style_Transfer_Workspace'

if not STYLE_WORKSPACE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', STYLE_WORKSPACE_REPO, str(STYLE_WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(STYLE_WORKSPACE), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(STYLE_WORKSPACE), 'reset', '--hard', 'origin/main'], check=True)

STYLE_DIR = STYLE_WORKSPACE / 'style'

# These start with simple ImageNet-friendly subjects and semantically compatible style references.
PAPER_CASES = [
    {
        'name': 'golden retriever + dog sketch',
        'prompt': 'a photo of a golden retriever sitting on grass',
        'style_path': STYLE_DIR / 'Sketch' / 'S005.png',
        'style_label': 'dog pencil sketch',
    },
    {
        'name': 'golden retriever + dog pixel art',
        'prompt': 'a photo of a golden retriever sitting on grass',
        'style_path': STYLE_DIR / 'PixelArt' / 'PA012.png',
        'style_label': 'dog pixel art',
    },
    {
        'name': 'mountain lake + watercolor landscape',
        'prompt': 'a photo of a mountain lake under a blue sky',
        'style_path': STYLE_DIR / 'WaterColor' / 'WC001.png',
        'style_label': 'watercolor landscape',
    },
    {
        'name': 'mountain lake + Monet landscape',
        'prompt': 'a photo of a mountain lake under a blue sky',
        'style_path': STYLE_DIR / 'Monet' / 'Monet001.png',
        'style_label': 'Monet landscape',
    },
]

for case in PAPER_CASES:
    assert case['style_path'].exists(), case['style_path']

CASE_ID = 0
case = PAPER_CASES[CASE_ID]
TARGET_PROMPT = case['prompt']
STYLE_REFERENCE_PATH = case['style_path']
print('case:', case['name'])
print('prompt:', TARGET_PROMPT)
print('style:', STYLE_REFERENCE_PATH)

plt.figure(figsize=(3, 3))
plt.imshow(Image.open(STYLE_REFERENCE_PATH).convert('RGB'))
plt.title(case['style_label'])
plt.axis('off')
plt.show()


## Style Feature Extraction and PFB

The frozen VAR VAE converts the style reference into cumulative multi-scale feature maps `F_style[s]`.

For any feature map `F = U Sigma V^T`, the paper’s extractor applies exponential singular-value weighting:

```text
Phi(F) = U W Sigma V^T
W_ii = exp(-i * alpha)
```

The default `alpha=1.0` is the paper setting. The rank and alpha sweeps below test whether VAR-CLIP has the same spectral behavior as Infinity.


In [ ]:
def load_style_reference(path, size=256):
    image = Image.open(path).convert('RGB')
    image = ImageOps.fit(image, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    image_01 = torchvision.transforms.functional.to_tensor(image).unsqueeze(0).to(device)
    image_m11 = image_01.mul(2).sub(1)
    return image_m11, image_01, image


@torch.no_grad()
def image_to_fhat_scales(image_m11):
    latent = vae.quant_conv(vae.encoder(image_m11))
    return vae.quantize.f_to_idxBl_or_fhat(latent, to_fhat=True)


def phi_principal_feature(feature_bchw, alpha=1.0, rank=None):
    """Phi(F): SVD reconstruction with exponentially weighted singular values."""
    dtype = feature_bchw.dtype
    batch, channels, height, width = feature_bchw.shape
    principal_features = []

    for batch_id in range(batch):
        feature_nc = feature_bchw[batch_id].detach().float().permute(1, 2, 0).reshape(height * width, channels)
        u, singular_values, vh = torch.linalg.svd(feature_nc, full_matrices=False)
        used_rank = singular_values.numel() if rank is None else min(rank, singular_values.numel())
        weights = torch.exp(-alpha * torch.arange(used_rank, dtype=feature_nc.dtype, device=feature_nc.device))
        principal_nc = (u[:, :used_rank] * (singular_values[:used_rank] * weights).unsqueeze(0)) @ vh[:used_rank]
        principal_features.append(principal_nc.reshape(height, width, channels).permute(2, 0, 1))

    return torch.stack(principal_features).to(dtype=dtype)


def edit_feature(generated_feature, style_feature, mode='pfb', alpha=1.0, rank=None):
    if mode == 'none':
        return generated_feature
    if mode == 'full_replace':
        return style_feature.to(dtype=generated_feature.dtype, device=generated_feature.device)
    if mode == 'pfb':
        style_feature = style_feature.to(dtype=generated_feature.dtype, device=generated_feature.device)
        return phi_principal_feature(style_feature, alpha=alpha, rank=rank) + (
            generated_feature - phi_principal_feature(generated_feature, alpha=alpha, rank=rank)
        )
    raise ValueError(f'Unknown edit mode: {mode}')


style_m11, style_01, style_pil = load_style_reference(STYLE_REFERENCE_PATH)
with torch.inference_mode():
    style_fhat_scales = image_to_fhat_scales(style_m11)
print('style feature shapes:', [tuple(feature.shape) for feature in style_fhat_scales])


## Paper-Faithful Dual-Stream VAR-CLIP Inference

`content` below means **the paper’s unmodified prompt-conditioned stream**, not a real content image.

Two separate random generators begin with the same seed. Consequently, both streams produce exactly the same tokens until PFB changes the stylized stream. This makes the unmodified stream a stable structural reference for SAC.

SAC is applied from the transformer update after PFB. In zero-indexed code, `PFB_STEP=2` means the third feature and `SAC_START_STEP=3` means the fourth transformer update.


In [ ]:
class _SACController:
    def __init__(self):
        self.mode = 'off'  # 'capture' for unmodified prompt path; 'inject' for stylized path
        self.content_qk = {}

    def clear_step(self):
        self.content_qk.clear()


def _sac_attention_forward(attention, x, attn_bias):
    batch, length, channels = x.shape
    qkv = F.linear(
        input=x,
        weight=attention.mat_qkv.weight,
        bias=torch.cat((attention.q_bias, attention.zero_k_bias, attention.v_bias)),
    ).view(batch, length, 3, attention.num_heads, attention.head_dim)

    # Force B,H,L,d layout so Q/K can be replaced before cached keys are appended.
    q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(dim=0)
    if attention.attn_l2_norm:
        scale_multiplier = attention.scale_mul_1H11.clamp_max(attention.max_scale_mul).exp()
        q = F.normalize(q, dim=-1).mul(scale_multiplier)
        k = F.normalize(k, dim=-1)

    controller = getattr(attention, '_sac_controller', None)
    if controller is not None and controller.mode == 'capture':
        controller.content_qk[attention.block_idx] = (q.detach(), k.detach())
    elif controller is not None and controller.mode == 'inject':
        content_q, content_k = controller.content_qk[attention.block_idx]
        if content_q.shape != q.shape or content_k.shape != k.shape:
            raise RuntimeError(f'SAC Q/K mismatch at transformer block {attention.block_idx}.')
        q = content_q.to(dtype=q.dtype, device=q.device)
        k = content_k.to(dtype=k.dtype, device=k.device)

    # SAC alters K only; V remains from the stylized generation stream.
    if attention.caching:
        if attention.cached_k is None:
            attention.cached_k, attention.cached_v = k, v
        else:
            attention.cached_k = torch.cat((attention.cached_k, k), dim=2)
            attention.cached_v = torch.cat((attention.cached_v, v), dim=2)
        k, v = attention.cached_k, attention.cached_v

    output = slow_attn(
        query=q,
        key=k,
        value=v,
        scale=attention.scale,
        attn_mask=attn_bias,
        dropout_p=attention.attn_drop if attention.training else 0.0,
    ).transpose(1, 2).reshape(batch, length, channels)
    return attention.proj_drop(attention.proj(output))


class _SACAttentionPatch:
    def __init__(self, model, controller):
        self.model = model
        self.controller = controller
        self.original_forwards = []

    def __enter__(self):
        for block in self.model.blocks:
            attention = block.attn
            self.original_forwards.append((attention, attention.forward))
            attention._sac_controller = self.controller
            attention.forward = types.MethodType(_sac_attention_forward, attention)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        for attention, original_forward in self.original_forwards:
            attention.forward = original_forward
            if hasattr(attention, '_sac_controller'):
                delattr(attention, '_sac_controller')
        return False


def _snapshot_cache(model):
    return [(block.attn.cached_k, block.attn.cached_v) for block in model.blocks]


def _restore_cache(model, state):
    for block, (cached_k, cached_v) in zip(model.blocks, state):
        block.attn.cached_k = cached_k
        block.attn.cached_v = cached_v


def _disable_cache(model):
    for block in model.blocks:
        block.attn.kv_caching(False)


def _prompt_condition(model, prompt_embedding):
    batch = prompt_embedding.shape[0]
    null_embedding = model.noise(torch.tensor(0, device=prompt_embedding.device)).unsqueeze(0).expand(batch, -1)
    return model.cond_proj(torch.cat((prompt_embedding, null_embedding), dim=0))


def _next_tokens_from_fhat(model, feature, next_step, level_position, current_length):
    patch_num = model.patch_nums[next_step]
    tokens = F.interpolate(feature, size=(patch_num, patch_num), mode='area')
    tokens = tokens.view(tokens.shape[0], model.Cvae, -1).transpose(1, 2)
    tokens = model.word_embed(tokens) + level_position[:, current_length:current_length + patch_num * patch_num]
    return tokens.repeat(2, 1, 1)  # conditional and classifier-free null branches


@torch.no_grad()
def run_dual_stream_pfb_sac(
    model,
    prompt_embedding,
    style_features,
    *,
    seed=0,
    cfg=4.0,
    top_k=900,
    top_p=0.95,
    pfb_step=2,
    sac_start_step=3,
    edit_mode='pfb',
    pfb_alpha=1.0,
    pfb_rank=None,
    enable_sac=True,
):
    """Paper-faithful dual prompt streams: same prompt, same seed, one style reference."""
    batch = prompt_embedding.shape[0]
    if batch != 1:
        raise ValueError('This research implementation currently uses one generated image at a time (B=1).')
    if not 0 <= pfb_step < len(model.patch_nums):
        raise ValueError('Invalid PFB step.')
    if enable_sac and not 0 <= sac_start_step < len(model.patch_nums):
        raise ValueError('Invalid SAC start step.')

    model.eval()
    content_rng = torch.Generator(device=prompt_embedding.device).manual_seed(seed)
    generation_rng = torch.Generator(device=prompt_embedding.device).manual_seed(seed)
    condition = _prompt_condition(model, prompt_embedding)
    level_position = model.lvl_embed(model.lvl_1L) + model.pos_1LC
    initial_tokens = (
        condition.unsqueeze(1).expand(2 * batch, model.first_l, -1)
        + model.pos_start.expand(2 * batch, model.first_l, -1)
        + level_position[:, :model.first_l]
    )

    content_tokens = initial_tokens
    generation_tokens = initial_tokens
    content_fhat = condition.new_zeros(batch, model.Cvae, model.patch_nums[-1], model.patch_nums[-1])
    generation_fhat = condition.new_zeros(batch, model.Cvae, model.patch_nums[-1], model.patch_nums[-1])
    content_trace, generation_trace = [], []

    controller = _SACController()
    _disable_cache(model)
    for block in model.blocks:
        block.attn.kv_caching(True)
    content_cache = _snapshot_cache(model)
    generation_cache = _snapshot_cache(model)

    try:
        with _SACAttentionPatch(model, controller):
            current_length = 0
            for step_id, patch_num in enumerate(model.patch_nums):
                current_length += patch_num * patch_num
                condition_for_blocks = model.shared_ada_lin(condition)

                # Unmodified content stream: regular VAR-CLIP inference under the same prompt.
                _restore_cache(model, content_cache)
                controller.clear_step()
                controller.mode = 'capture'
                content_hidden = content_tokens
                for block in model.blocks:
                    content_hidden = block(x=content_hidden, cond_BD=condition_for_blocks, attn_bias=None)
                content_logits = model.get_logits(content_hidden, condition)
                cfg_ratio = cfg * (step_id / model.num_stages_minus_1)
                content_logits = (1 + cfg_ratio) * content_logits[:batch] - cfg_ratio * content_logits[batch:]
                content_indices = sample_with_top_k_top_p_(
                    content_logits, rng=content_rng, top_k=top_k, top_p=top_p, num_samples=1
                )[:, :, 0]
                content_residual = model.vae_quant_proxy[0].embedding(content_indices).transpose(1, 2).reshape(
                    batch, model.Cvae, patch_num, patch_num
                )
                content_fhat, _ = model.vae_quant_proxy[0].get_next_autoregressive_input(
                    step_id, len(model.patch_nums), content_fhat, content_residual
                )
                content_cache = _snapshot_cache(model)

                # Stylized generation stream: it stays identical until PFB edits its feature.
                _restore_cache(model, generation_cache)
                controller.mode = 'inject' if enable_sac and step_id >= sac_start_step else 'off'
                generation_hidden = generation_tokens
                for block in model.blocks:
                    generation_hidden = block(x=generation_hidden, cond_BD=condition_for_blocks, attn_bias=None)
                generation_logits = model.get_logits(generation_hidden, condition)
                generation_logits = (1 + cfg_ratio) * generation_logits[:batch] - cfg_ratio * generation_logits[batch:]
                generation_indices = sample_with_top_k_top_p_(
                    generation_logits, rng=generation_rng, top_k=top_k, top_p=top_p, num_samples=1
                )[:, :, 0]
                generation_residual = model.vae_quant_proxy[0].embedding(generation_indices).transpose(1, 2).reshape(
                    batch, model.Cvae, patch_num, patch_num
                )
                generation_fhat, _ = model.vae_quant_proxy[0].get_next_autoregressive_input(
                    step_id, len(model.patch_nums), generation_fhat, generation_residual
                )

                if edit_mode != 'none' and step_id == pfb_step:
                    generation_fhat = edit_feature(
                        generation_fhat,
                        style_features[step_id],
                        mode=edit_mode,
                        alpha=pfb_alpha,
                        rank=pfb_rank,
                    )

                content_trace.append(content_fhat.detach().clone())
                generation_trace.append(generation_fhat.detach().clone())
                generation_cache = _snapshot_cache(model)

                if step_id != model.num_stages_minus_1:
                    content_tokens = _next_tokens_from_fhat(
                        model, content_fhat, step_id + 1, level_position, current_length
                    )
                    generation_tokens = _next_tokens_from_fhat(
                        model, generation_fhat, step_id + 1, level_position, current_length
                    )

        return {
            'content_image_01': model.vae_proxy[0].fhat_to_img(content_fhat).add(1).mul(0.5),
            'stylized_image_01': model.vae_proxy[0].fhat_to_img(generation_fhat).add(1).mul(0.5),
            'content_fhat_scales': content_trace,
            'stylized_fhat_scales': generation_trace,
        }
    finally:
        controller.mode = 'off'
        _disable_cache(model)


## Main Paper Configuration and Component Ablation

This is the first cell to run after setup.

It compares:

```text
baseline        : ordinary VAR-CLIP, no feature edit
full replacement: replace F_s^gen with the entire style feature F_s^style
PFB only        : replace only principal style components
PFB + SAC       : PFB plus prompt-stream structural Q/K correction
```

Full replacement is intentionally expected to leak the style-reference subject. It is the negative control that shows why PFB is needed.


In [ ]:
# Paper default translated to VAR-CLIP's ten-scale pyramid.
PFB_STEP = 2           # third cumulative feature F_3
SAC_START_STEP = 3      # next transformer update after F_3 is edited
PFB_ALPHA = 1.0         # paper default
PFB_RANK = None         # full exponential spectrum for the main method
CFG = 4.0
TOP_K = 900
TOP_P = 0.95
SEED = 0
RUN_COMPONENT_ABLATION = True


def prepare_prompt_embedding(prompt):
    tokens = tokenize([prompt]).to(device)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        return clip_model.encode_text(tokens)


def show_images(items, title, columns=5):
    rows = math.ceil(len(items) / columns)
    plt.figure(figsize=(4 * columns, 4 * rows))
    for index, (name, image) in enumerate(items):
        plt.subplot(rows, columns, index + 1)
        image = image.detach().float().cpu()
        if image.ndim == 4:
            image = image[0]
        plt.imshow(image.clamp(0, 1).permute(1, 2, 0).numpy())
        plt.title(name)
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


prompt_embedding = prepare_prompt_embedding(TARGET_PROMPT)

if RUN_COMPONENT_ABLATION:
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        baseline = run_dual_stream_pfb_sac(
            var_clip, prompt_embedding, style_fhat_scales,
            seed=SEED, cfg=CFG, top_k=TOP_K, top_p=TOP_P,
            pfb_step=PFB_STEP, sac_start_step=SAC_START_STEP,
            edit_mode='none', enable_sac=False,
        )
        full_replacement = run_dual_stream_pfb_sac(
            var_clip, prompt_embedding, style_fhat_scales,
            seed=SEED, cfg=CFG, top_k=TOP_K, top_p=TOP_P,
            pfb_step=PFB_STEP, sac_start_step=SAC_START_STEP,
            edit_mode='full_replace', enable_sac=False,
        )
        pfb_only = run_dual_stream_pfb_sac(
            var_clip, prompt_embedding, style_fhat_scales,
            seed=SEED, cfg=CFG, top_k=TOP_K, top_p=TOP_P,
            pfb_step=PFB_STEP, sac_start_step=SAC_START_STEP,
            edit_mode='pfb', pfb_alpha=PFB_ALPHA, pfb_rank=PFB_RANK, enable_sac=False,
        )
        pfb_sac = run_dual_stream_pfb_sac(
            var_clip, prompt_embedding, style_fhat_scales,
            seed=SEED, cfg=CFG, top_k=TOP_K, top_p=TOP_P,
            pfb_step=PFB_STEP, sac_start_step=SAC_START_STEP,
            edit_mode='pfb', pfb_alpha=PFB_ALPHA, pfb_rank=PFB_RANK, enable_sac=True,
        )

    component_items = [
        (f'style reference\n{case["style_label"]}', style_01),
        (f'content stream\n"{TARGET_PROMPT}"', baseline['content_image_01']),
        ('VAR-CLIP baseline', baseline['stylized_image_01']),
        ('full feature replacement', full_replacement['stylized_image_01']),
        ('PFB only', pfb_only['stylized_image_01']),
        ('PFB + SAC', pfb_sac['stylized_image_01']),
    ]
    show_images(component_items, f'{case["name"]} | paper-style component ablation', columns=3)

    component_path = OUTPUT_DIR / f'component_ablation_case{CASE_ID}_seed{SEED}.png'
    grid = torchvision.utils.make_grid(
        torch.cat([image for _, image in component_items], dim=0).detach().float().cpu().clamp(0, 1),
        nrow=3,
        padding=2,
        pad_value=1.0,
    )
    Image.fromarray(grid.permute(1, 2, 0).mul(255).byte().numpy()).save(component_path)
    print('saved:', component_path)


## Diagnostic Metrics

These are lightweight diagnostics for **comparisons within the same run**, not the paper’s exact benchmark metrics:

```text
prompt cosine:      generated image vs target prompt in the model's CLIP space
style-image cosine: generated image vs style reference in CLIP image space
content cosine:     generated image vs the unmodified content stream in CLIP image space
harmonic:           harmonic mean of prompt and style-image cosine
```

Style-image cosine is not pure style similarity because the reference image also has a subject. It is recorded only to compare ablations under the same style reference.


In [ ]:
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


def clip_image_embedding(image_01):
    image_01 = F.interpolate(image_01.to(device), size=(224, 224), mode='bicubic', align_corners=False)
    image_for_clip = (image_01.clamp(0, 1) - CLIP_MEAN) / CLIP_STD
    return clip_model.encode_image(image_for_clip)


def cosine_score(embedding_a, embedding_b):
    return float((embedding_a * embedding_b).sum(dim=-1).mean().detach().cpu())


def score_result(result, prompt_embedding, style_image_01):
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        generated_embedding = clip_image_embedding(result['stylized_image_01'])
        style_embedding = clip_image_embedding(style_image_01)
        content_embedding = clip_image_embedding(result['content_image_01'])
    prompt_score = cosine_score(generated_embedding, prompt_embedding)
    style_score = cosine_score(generated_embedding, style_embedding)
    content_score = cosine_score(generated_embedding, content_embedding)
    harmonic = 2 * prompt_score * style_score / max(prompt_score + style_score, 1e-8)
    return {
        'prompt_clip_cosine': prompt_score,
        'style_image_clip_cosine': style_score,
        'content_stream_clip_cosine': content_score,
        'prompt_style_harmonic': harmonic,
    }


if RUN_COMPONENT_ABLATION:
    component_metrics = pd.DataFrame([
        {'method': 'baseline', **score_result(baseline, prompt_embedding, style_01)},
        {'method': 'full_replace', **score_result(full_replacement, prompt_embedding, style_01)},
        {'method': 'pfb_only', **score_result(pfb_only, prompt_embedding, style_01)},
        {'method': 'pfb_sac', **score_result(pfb_sac, prompt_embedding, style_01)},
    ])
    display(component_metrics.round(4))
    metric_path = OUTPUT_DIR / f'component_metrics_case{CASE_ID}_seed{SEED}.csv'
    component_metrics.to_csv(metric_path, index=False)
    print('saved:', metric_path)


## Ablation A: Pivotal PFB Step

The paper uses the third feature because it was empirically pivotal for Infinity. This sweep asks the same question for VAR-CLIP-d16.

It costs ten dual-stream generations. Keep the prompt, reference, and seed fixed; vary only the injection step.


In [ ]:
RUN_PIVOTAL_STEP_ABLATION = False

if RUN_PIVOTAL_STEP_ABLATION:
    pivotal_results = []
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for step_id, patch_num in enumerate(tqdm(PATCH_NUMS, desc='Pivotal-step ablation')):
            result = run_dual_stream_pfb_sac(
                var_clip, prompt_embedding, style_fhat_scales,
                seed=SEED, cfg=CFG, top_k=TOP_K, top_p=TOP_P,
                pfb_step=step_id,
                sac_start_step=min(step_id + 1, len(PATCH_NUMS) - 1),
                edit_mode='pfb', pfb_alpha=PFB_ALPHA, pfb_rank=PFB_RANK,
                enable_sac=step_id < len(PATCH_NUMS) - 1,
            )
            pivotal_results.append((f'PFB step {step_id + 1}\n({patch_num}x{patch_num})', result))

    show_images(
        [(name, result['stylized_image_01']) for name, result in pivotal_results],
        f'{case["name"]} | VAR-CLIP pivotal-step ablation',
        columns=4,
    )
    pivotal_metrics = pd.DataFrame([
        {'pfb_step': index + 1, 'grid_size': PATCH_NUMS[index], **score_result(result, prompt_embedding, style_01)}
        for index, (_, result) in enumerate(pivotal_results)
    ])
    display(pivotal_metrics.round(4))
    pivotal_metrics.to_csv(OUTPUT_DIR / f'pivotal_step_metrics_case{CASE_ID}_seed{SEED}.csv', index=False)


## Ablation B: Number of Principal Singular Components

The paper evaluates top-`k` SVD components for `k = {1, 2, 4, 8, 16, 32}`. Its hypothesis is:

```text
small k  -> mostly style/color, lower content leakage
large k  -> more source structure and identity leak into the generated image
```

This test determines whether the same spectral separation exists in VAR-CLIP.


In [ ]:
RUN_RANK_ABLATION = False
RANKS_TO_TEST = [1, 2, 4, 8, 16, 32]

if RUN_RANK_ABLATION:
    rank_results = []
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for rank in tqdm(RANKS_TO_TEST, desc='Principal-rank ablation'):
            result = run_dual_stream_pfb_sac(
                var_clip, prompt_embedding, style_fhat_scales,
                seed=SEED, cfg=CFG, top_k=TOP_K, top_p=TOP_P,
                pfb_step=PFB_STEP, sac_start_step=SAC_START_STEP,
                edit_mode='pfb', pfb_alpha=PFB_ALPHA, pfb_rank=rank,
                enable_sac=True,
            )
            rank_results.append((f'top-{rank} singular components', result))

    show_images(
        [(name, result['stylized_image_01']) for name, result in rank_results],
        f'{case["name"]} | principal-component rank ablation',
        columns=3,
    )
    rank_metrics = pd.DataFrame([
        {'rank': rank, **score_result(result, prompt_embedding, style_01)}
        for rank, (_, result) in zip(RANKS_TO_TEST, rank_results)
    ])
    display(rank_metrics.round(4))
    rank_metrics.to_csv(OUTPUT_DIR / f'rank_metrics_case{CASE_ID}_seed{SEED}.csv', index=False)


## Ablation C: Exponential Decay Rate Alpha

The paper tests `alpha = {0.2, 0.6, 1.0, 2.0, 5.0}` and uses `alpha=1.0` as the balance point. Lower alpha gives higher-rank components more influence and can increase style strength, but also increases content leakage from the reference.


In [ ]:
RUN_ALPHA_ABLATION = False
ALPHAS_TO_TEST = [0.2, 0.6, 1.0, 2.0, 5.0]

if RUN_ALPHA_ABLATION:
    alpha_results = []
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for alpha in tqdm(ALPHAS_TO_TEST, desc='PFB alpha ablation'):
            result = run_dual_stream_pfb_sac(
                var_clip, prompt_embedding, style_fhat_scales,
                seed=SEED, cfg=CFG, top_k=TOP_K, top_p=TOP_P,
                pfb_step=PFB_STEP, sac_start_step=SAC_START_STEP,
                edit_mode='pfb', pfb_alpha=alpha, pfb_rank=None,
                enable_sac=True,
            )
            alpha_results.append((f'alpha = {alpha}', result))

    show_images(
        [(name, result['stylized_image_01']) for name, result in alpha_results],
        f'{case["name"]} | PFB exponential-decay ablation',
        columns=3,
    )
    alpha_metrics = pd.DataFrame([
        {'alpha': alpha, **score_result(result, prompt_embedding, style_01)}
        for alpha, (_, result) in zip(ALPHAS_TO_TEST, alpha_results)
    ])
    display(alpha_metrics.round(4))
    alpha_metrics.to_csv(OUTPUT_DIR / f'alpha_metrics_case{CASE_ID}_seed{SEED}.csv', index=False)


## Optional: Curated Prompt/Style Case Suite

After selecting a stable setting from the ablations, run the same method on the four easy, compatible cases. This is a qualitative robustness check; it is not a replacement for a benchmark.


In [ ]:
RUN_CURATED_CASE_SUITE = False

if RUN_CURATED_CASE_SUITE:
    suite_items = []
    suite_metrics = []
    for current_case_id, current_case in enumerate(tqdm(PAPER_CASES, desc='Curated prompt/style cases')):
        current_style_m11, current_style_01, _ = load_style_reference(current_case['style_path'])
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            current_style_features = image_to_fhat_scales(current_style_m11)
            current_prompt_embedding = prepare_prompt_embedding(current_case['prompt'])
            current_result = run_dual_stream_pfb_sac(
                var_clip, current_prompt_embedding, current_style_features,
                seed=SEED, cfg=CFG, top_k=TOP_K, top_p=TOP_P,
                pfb_step=PFB_STEP, sac_start_step=SAC_START_STEP,
                edit_mode='pfb', pfb_alpha=PFB_ALPHA, pfb_rank=PFB_RANK,
                enable_sac=True,
            )
        suite_items.extend([
            (f'{current_case["name"]}\nstyle reference', current_style_01),
            (f'{current_case["name"]}\nPFB + SAC', current_result['stylized_image_01']),
        ])
        suite_metrics.append({
            'case_id': current_case_id,
            'case': current_case['name'],
            **score_result(current_result, current_prompt_embedding, current_style_01),
        })
        torch.cuda.empty_cache()

    show_images(suite_items, 'Curated VAR-CLIP prompt/style suite', columns=2)
    suite_metrics = pd.DataFrame(suite_metrics)
    display(suite_metrics.round(4))
    suite_metrics.to_csv(OUTPUT_DIR / f'curated_suite_metrics_seed{SEED}.csv', index=False)


## How To Interpret the Study

```text
Baseline is coherent, PFB changes texture/color, PFB+SAC restores shape:
    the paper mechanism transfers meaningfully to VAR-CLIP.

Full replacement imports the source animal/landscape while PFB does not:
    PFB is suppressing source-subject leakage as intended.

Rank 1-2 works best and larger ranks distort the object:
    VAR-CLIP has a similar style-versus-structure spectral split.

No rank or alpha produces a good result:
    the Infinity pivotal-feature hypothesis does not transfer to VAR-CLIP-d16.

A later PFB step wins the scale sweep:
    use the VAR-CLIP-specific step; do not keep the paper's third-step setting by habit.
```

Run order:

```text
1. Main component ablation on the dog-sketch case.
2. Pivotal-step ablation.
3. Rank ablation at the best step.
4. Alpha ablation at the best step/rank.
5. Curated suite with the selected setting.
```
